<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [ ]:
import os
from ui_lib import *

# Paths
model_path = "Body_detection_model.pt"

input_video_directory = "input"
output_video_directory = "output"
temp_directory = f"{output_video_directory}/temp"
raw_text_output_directory = f"{temp_directory}/raw_output"
manual_annotations_directory = f"{input_video_directory}/manual_annotations"
treated_directory = f"{output_video_directory}/treated"
final_directory = f"{output_video_directory}/final"

# Videos to ignore per step
ignore_S1 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big",
    "20241019 - 13h28-(temp)",
    "20241019 - 13h28",
    "20241019 - 14h29-(temp)",
    "20241019 - 14h29",
]

ignore_S2 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big",
    "20241019 - 13h28-(temp)",
    "20241019 - 13h28",
    "20241019 - 14h29-(temp)",
    "20241019 - 14h29",
]

ignore_S3 = [
    "example_1",
    "example_2",
    "20241009 - 09h07.MP4",
    "20241009 - 09h07",
    "big.MP4",
    "big",
]

# Helper: create required folders
for path in [
    input_video_directory,
    output_video_directory,
    temp_directory,
    raw_text_output_directory,
    manual_annotations_directory,
    treated_directory,
    final_directory,
]:
    os.makedirs(path, exist_ok=True)

# DeepSORT/YOLO setup
max_cosine_distance = 0.5
nn_budget = None
metric = nn_matching.NearestNeighborDistanceMetric("cosine", max_cosine_distance, nn_budget)

YOLOv8s = YOLO(model_path)
DeepSort = DeepSortTracker(metric)
Osnet = torchreid.models.build_model(name="osnet_x1_0", num_classes=751, pretrained=True)
Osnet.eval()

def has_audio_stream(video_path: str) -> bool:
    """
    Optional: quick check to avoid mux when there is no audio.
    If your mux_audio already tolerates missing audio, you can skip this.
    """
    try:
        import subprocess, json
        probe_cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json",
            video_path,
        ]
        out = subprocess.check_output(probe_cmd).decode("utf-8")
        data = json.loads(out)
        streams = data.get("streams", [])
        return len(streams) > 0
    except Exception:
        # Fallback: assume audio exists to keep behavior; mux_audio should handle errors gracefully
        return True

/home/diego/.pyenv/versions/chimprec310/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(
2026-02-09 18:07:02.269174: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-09 18:07:02.430685: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-09 18:07:02.473102: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic 

Successfully loaded imagenet pretrained weights from "/home/diego/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file_path = f"{manual_annotations_directory}/{video_name}.txt"
    try:
        with open(annotation_file_path, "x") as f:
            print(f"{video_name}.txt automatically created in {manual_annotations_directory}.")
    except FileExistsError:
        print(f"{video_name}.txt already present in {manual_annotations_directory}.")
    print()

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"
    perform_tracking(
        input_video_path=full_video_path,
        output_text_file_path=raw_txt_path,
        detection_model=YOLOv8s,
        tracker=DeepSort,
        confidence_threshold=0.5,
        model_feature_extraction=Osnet,
    )
    print(f"Annotations ready for video: {full_video_path}.\n")

    processed_with_audio = f"{temp_directory}/{video_name}-(temp)-audio.mp4"

    # Draw and mux (only one output, with audio when available)
    draw_bbox_from_file(
        file_path=raw_txt_path,
        input_video_path=full_video_path,
        output_video_path=processed_with_audio,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, processed_with_audio, processed_with_audio)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored
20241015 - 12h41-(tempCut).txt already present in input/manual_annotations.



Tracking progress (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:43<00:00, 11.02it/s]


Annotations ready for video: input/20241015 - 12h41-(tempCut).mp4.



Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:08<00:00, 60.06it/s]
ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20260103
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enabl

Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.

big.mp4 ignored
20241019 - 14h29-(temp).mp4 ignored
20241019 - 13h28.txt already present in input/manual_annotations.



Tracking progress (20241019 - 13h28.MP4):   0%|          | 118/55128 [00:09<1:12:41, 12.61it/s]


KeyboardInterrupt: 

<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [2]:
# ---------- STEP 2: apply manual edits -> treated output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-treated.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored
20241015 - 12h41-(tempCut).mp4 ignored
big.mp4 ignored
20241019 - 13h28-(temp).mp4 ignored


Drawing annotations (20241019 - 14h29-(temp).mp4): 100%|██████████| 55032/55032 [15:26<00:00, 59.42it/s]


No audio stream detected; skipping mux.
Treatment done: input/20241019 - 14h29-(temp).mp4.



<h2> Third Step: </h2>

This final step will only be used to generate the arrows associated with the corresponding chimpanzees.<br>
<b>It's only to be performed when the manual annotations are 100% correct.</b>


In [4]:
# ---------- STEP 3: final arrows/names -> final output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{manual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-final.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="triangle",
        draw_frame_count=False,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241009 - 09h07.mp4 ignored


Drawing annotations (20241015 - 12h41-(tempCut).mp4): 100%|██████████| 484/484 [00:08<00:00, 58.92it/s]
ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enabl

Adding audio...
Treatment done: input/20241015 - 12h41-(tempCut).mp4.



Drawing annotations (big.MP4):   0%|          | 27/16392 [00:00<04:16, 63.86it/s]
ffmpeg version n8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.1 (GCC) 20251112
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-l

Adding audio...
Treatment done: input/big.MP4.



[out#0/mp4 @ 0x55d914bce800] video:1218KiB audio:61470KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.003895%
frame=   27 fps=0.0 q=-1.0 Lsize=   62691KiB time=00:00:00.54 bitrate=951038.8kbits/s speed= 4.3x elapsed=0:00:00.12    
